# Ohm Ganesha

In [1]:
import os, sys, pyspark

# Override SPARK_HOME before JVM starts — must be FIRST cell
os.environ["SPARK_HOME"]            = pyspark.__path__[0]
os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Confirm — SPARK_HOME must NOT contain 'homebrew' or 'apache-spark'
assert "homebrew" not in os.environ["SPARK_HOME"].lower(), "❌ Still pointing to Homebrew!"
assert "apache-spark" not in os.environ["SPARK_HOME"].lower(), "❌ Still pointing to Homebrew!"
print(f"✅ SPARK_HOME clean: {os.environ['SPARK_HOME']}")
print(f"✅ Python: {sys.version.split()[0]}, PySpark: {pyspark.__version__}")


✅ SPARK_HOME clean: /Users/vijaymurthyms/projects/iceberg/iceberg-mastery/.venv/lib/python3.11/site-packages/pyspark
✅ Python: 3.11.14, PySpark: 3.5.3


In [2]:
from pyspark.sql import SparkSession

PROJECT_ROOT   = "/Users/vijaymurthyms/projects/iceberg/iceberg-mastery"
WAREHOUSE_PATH = os.path.join(PROJECT_ROOT, "warehouse")

spark = (
    SparkSession.builder
    .appName("IcebergMastery")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.0")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type",      "hadoop")
    .config("spark.sql.catalog.local.warehouse", WAREHOUSE_PATH)
    .config("spark.driver.host",        "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.executor.processTreeMetrics.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark: {spark.version}")   # Must say 3.5.3


:: loading settings :: url = jar:file:/Users/vijaymurthyms/projects/iceberg/iceberg-mastery/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/vijaymurthyms/.ivy2/cache
The jars for the packages stored in: /Users/vijaymurthyms/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2a7f46fb-ad70-4869-b5b1-b119b95b6a7f;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.0 in central
:: resolution report :: resolve 44ms :: artifacts dl 1ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :

✅ Spark: 3.5.3


In [3]:
# Step 2: Create a table with NO data — watch what files appear
spark.sql("DROP TABLE IF EXISTS local.db.orders")
spark.sql("""
    CREATE TABLE local.db.orders (
        order_id   INT,
        customer   STRING,
        amount     DOUBLE,
        order_date DATE
    )
    USING iceberg
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.parquet.compression-codec' = 'snappy'
    )
""")
print("✅ Table created. Now look at warehouse/db/orders/ in VS Code!")


✅ Table created. Now look at warehouse/db/orders/ in VS Code!


warehouse/
└── db/
    └── orders/
        └── metadata/
            ├── version-hint.txt        ← "1" (just the number 1)
            └── v1.metadata.json        ← The table's constitution


In [4]:
# Step 3: Insert data and watch ALL files appear
from datetime import date

data = [
    (1, "Alice",   250.0, date(2024, 1, 15)),
    (2, "Bob",     180.0, date(2024, 2, 20)),
    (3, "Charlie", 320.0, date(2024, 1, 10)),
]

df = spark.createDataFrame(data, "order_id INT, customer STRING, amount DOUBLE, order_date DATE")
df.writeTo("local.db.orders").append()

print("✅ Data written. Refresh warehouse/ in VS Code now!")


✅ Data written. Refresh warehouse/ in VS Code now!


warehouse/
└── db/
    └── orders/
        ├── metadata/
        │   ├── version-hint.txt              ← Now says "2"
        │   ├── v1.metadata.json              ← Original (unchanged, immutable)
        │   ├── v2.metadata.json              ← New version pointing to snapshot
        │   └── snap-7823641092834.avro       ← Manifest List (snapshot file)
        │       └── [inside: points to manifest files]
        └── data/
            └── 00000-1-abc123.parquet        ← Your actual 3 rows of data!


In [9]:
df= spark.sql("SELECT * FROM local.db.orders")
df.show(10, False)

+--------+--------+------+----------+
|order_id|customer|amount|order_date|
+--------+--------+------+----------+
|1       |Alice   |250.0 |2024-01-15|
|2       |Bob     |180.0 |2024-02-20|
|3       |Charlie |320.0 |2024-01-10|
+--------+--------+------+----------+



In [5]:
# Step 3b: Use Iceberg's metadata tables to inspect everything
print("=" * 60)
print("📸 SNAPSHOTS TABLE")
spark.sql("""
    SELECT snapshot_id, committed_at, operation, 
           summary['added-records']     AS rows_added,
           summary['added-data-files']  AS files_added
    FROM local.db.orders.snapshots
""").show(truncate=False)

print("=" * 60)
print("📋 MANIFESTS TABLE")
spark.sql("""
    SELECT path, length, added_data_files_count, 
           existing_data_files_count, partition_summaries
    FROM local.db.orders.manifests
""").show(truncate=False)

print("=" * 60)
print("📁 FILES TABLE (actual Parquet files)")
spark.sql("""
    SELECT file_path, file_format, record_count, 
           file_size_in_bytes, column_sizes
    FROM local.db.orders.files
""").show(truncate=False)

print("=" * 60)
print("📚 METADATA LOG (version history)")
spark.sql("""
    SELECT * FROM local.db.orders.metadata_log_entries
""").show(truncate=False)


📸 SNAPSHOTS TABLE
+------------------+-----------------------+---------+----------+-----------+
|snapshot_id       |committed_at           |operation|rows_added|files_added|
+------------------+-----------------------+---------+----------+-----------+
|549342836411795743|2026-03-10 23:11:48.563|append   |3         |3          |
+------------------+-----------------------+---------+----------+-----------+

📋 MANIFESTS TABLE
+-------------------------------------------------------------------------------------------------------------------------------+------+----------------------+-------------------------+-------------------+
|path                                                                                                                           |length|added_data_files_count|existing_data_files_count|partition_summaries|
+-------------------------------------------------------------------------------------------------------------------------------+------+----------------------+--

In [8]:
# Step 3b: Use Iceberg's metadata tables to inspect everything
print("=" * 60)
print("📸 SNAPSHOTS TABLE")
spark.sql("""
    SELECT *
    FROM local.db.orders.snapshots
""").show(truncate=False)

print("=" * 60)
print("📋 MANIFESTS TABLE")
spark.sql("""
    SELECT *
    FROM local.db.orders.manifests
""").show(truncate=False)

print("=" * 60)
print("📁 FILES TABLE (actual Parquet files)")
spark.sql("""
    SELECT *
    FROM local.db.orders.files
""").show(truncate=False)

print("=" * 60)
print("📚 METADATA LOG (version history)")
spark.sql("""
    SELECT * FROM local.db.orders.metadata_log_entries
""").show(truncate=False)


📸 SNAPSHOTS TABLE
+-----------------------+------------------+---------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id       |parent_id|operation|manifest_list                                                                                                                                         |summary                                                                                      

In [10]:
# Step 4: Write more data — a 2nd snapshot is created
more_data = [(4, "Diana", 99.0, date(2024, 3, 5)),
             (5, "Eve",  410.0, date(2024, 3, 8))]
df2 = spark.createDataFrame(more_data, "order_id INT, customer STRING, amount DOUBLE, order_date DATE")
df2.writeTo("local.db.orders").append()

# Now version-hint.txt says "3" and v3.metadata.json exists
# You'll see TWO snap-XXXXX.avro files in metadata/

print("📸 Snapshot chain — notice parent_id linking them like a linked list:")
spark.sql("""
    SELECT snapshot_id, parent_id, committed_at, operation
    FROM local.db.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)


📸 Snapshot chain — notice parent_id linking them like a linked list:
+-------------------+------------------+-----------------------+---------+
|snapshot_id        |parent_id         |committed_at           |operation|
+-------------------+------------------+-----------------------+---------+
|549342836411795743 |NULL              |2026-03-10 23:11:48.563|append   |
|7192975603116258646|549342836411795743|2026-03-10 23:39:49.702|append   |
+-------------------+------------------+-----------------------+---------+



metadata/
├── version-hint.txt         ← "3"
├── v1.metadata.json         ← CREATE TABLE  (never changed)
├── v2.metadata.json         ← After 1st INSERT
├── v3.metadata.json         ← After 2nd INSERT (current)
├── snap-111111.avro         ← Manifest List for snapshot 1
└── snap-222222.avro         ← Manifest List for snapshot 2


data/
├── 00000-1-abc123.parquet   ← 3 rows from 1st insert
└── 00000-2-def456.parquet   ← 2 rows from 2nd insert


In [13]:
spark.sql("select * from local.db.orders.files").show()

+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|    readable_metrics|
+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|      0|/Users/vijaymurth...|    PARQUET|      0|           1|              1151|{1 -> 35, 2 -> 40...|{1 -> 1, 2 -> 1, ...|{1 -> 0, 2 -> 0, ...|        {3 -> 0

In [11]:
#Step 5 — File Format Deep Dive

In [15]:
# Step 5: Understand what's INSIDE each file type

# ── Parquet (data files) ──────────────────────────────────────
# Binary columnar format. You can't open it in VS Code directly,
# but Spark can tell you everything about it:
spark.sql("""
    SELECT 
        split(file_path, '/')[-1]   AS filename,
        file_format,
        record_count,
        file_size_in_bytes,
        column_sizes,               -- bytes per column
        value_counts,               -- non-null count per column  
        null_value_counts,
        lower_bounds,               -- min value per column (used for pruning!)
        upper_bounds                -- max value per column (used for pruning!)
    FROM local.db.orders.files
""").show(truncate=False)

# ── Avro (manifest files) ─────────────────────────────────────
# Binary format, stores manifest data. Contains file stats.
# You CAN read it with Python's fastavro library:
# pip install fastavro
import fastavro
import glob

manifest_files = glob.glob(f"{WAREHOUSE_PATH}/db/orders/metadata/*.avro")
for mf in manifest_files[:1]:   # read first one
    with open(mf, 'rb') as f:
        reader = fastavro.reader(f)
        print(f"\n📄 Avro Schema of {mf.split('/')[-1]}:")
        print(reader.writer_schema)
        print("\nEntries:")
        for record in reader:
            print(record)


+--------+-----------+------------+------------------+------------------------------------+--------------------------------+--------------------------------+-----------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------+
|filename|file_format|record_count|file_size_in_bytes|column_sizes                        |value_counts                    |null_value_counts               |lower_bounds                                                                                         |upper_bounds                                                                                         |
+--------+-----------+------------+------------------+------------------------------------+--------------------------------+--------------------------------+-----------------------------------------------------------------------------------------------------+-----------------

File	Format	Readable?	Purpose
v1.metadata.json	JSON	✅ Open in VS Code	Table blueprint: schema, partitions, snapshot list
snap-XXXX.avro	Avro	With fastavro	Manifest List: which manifest files belong to a snapshot
manifest-XXXX.avro	Avro	With fastavro	Manifest File: which Parquet files + their column stats
part-XXXX.parquet	Parquet (Snappy)	With pandas/spark	Actual data rows, columnar, compressed
version-hint.txt	Plain text	✅ Open in VS Code	Hadoop Catalog pointer: current metadata version number

In [17]:
# Step 6 — Partitioned Table: How the Folder Structure Changes  

In [23]:
# Step 6: Partitioned table — see how data/ folder changes
spark.sql("DROP TABLE IF EXISTS local.db.sales")
spark.sql("""
    CREATE TABLE local.db.sales (
        sale_id  INT,
        region   STRING,
        amount   DOUBLE,
        sale_date DATE
    )
    USING iceberg
    PARTITIONED BY (region)  
""")

# PARTITIONED BY (region)    # ← physical folder-per-region

spark.sql("SELECT * FROM local.db.sales").show()


+-------+------+------+---------+
|sale_id|region|amount|sale_date|
+-------+------+------+---------+
+-------+------+------+---------+



In [24]:
sales_data = [
    (1, "North", 500.0, date(2024, 1, 1)),
    (2, "South", 300.0, date(2024, 1, 1)),
    (3, "North", 700.0, date(2024, 1, 2)),
    (4, "East",  400.0, date(2024, 1, 3)),
]
df3 = spark.createDataFrame(sales_data, "sale_id INT, region STRING, amount DOUBLE, sale_date DATE")
df3.writeTo("local.db.sales").append()

warehouse/db/sales/
├── metadata/
│   ├── version-hint.txt
│   ├── v1.metadata.json
│   └── v2.metadata.json
└── data/
    ├── region=North/
    │   └── 00000-1-aaa.parquet    ← Only North rows (2 rows)
    ├── region=South/
    │   └── 00000-1-bbb.parquet    ← Only South rows (1 row)
    └── region=East/
        └── 00000-1-ccc.parquet    ← Only East rows (1 row)


This is how partition pruning works: a query WHERE region = 'North' makes Spark skip region=South/ and region=East/ entirely — it never even opens those Parquet files.

In [25]:
# Verify pruning — check the manifest to see partition stats
spark.sql("""
    SELECT partition, record_count, file_count
    FROM local.db.sales.partitions
""").show()


+---------+------------+----------+
|partition|record_count|file_count|
+---------+------------+----------+
|  {South}|           1|         1|
|  {North}|           2|         1|
|   {East}|           1|         1|
+---------+------------+----------+



In [27]:
# partition pruning — only files for 'North' read

# Spark will only read 1 Parquet file for this query (proven by explain plan)
spark.sql("SELECT * FROM local.db.sales WHERE region = 'North'").explain(True)
# Look for "PartitionFilters" in the explain output — that's pruning in action!

== Parsed Logical Plan ==
'Project [*]
+- 'Filter ('region = North)
   +- 'UnresolvedRelation [local, db, sales], [], false

== Analyzed Logical Plan ==
sale_id: int, region: string, amount: double, sale_date: date
Project [sale_id#1224, region#1225, amount#1226, sale_date#1227]
+- Filter (region#1225 = North)
   +- SubqueryAlias local.db.sales
      +- RelationV2[sale_id#1224, region#1225, amount#1226, sale_date#1227] local.db.sales local.db.sales

== Optimized Logical Plan ==
RelationV2[sale_id#1224, region#1225, amount#1226, sale_date#1227] local.db.sales

== Physical Plan ==
*(1) ColumnarToRow
+- BatchScan local.db.sales[sale_id#1224, region#1225, amount#1226, sale_date#1227] local.db.sales (branch=null) [filters=region IS NOT NULL, region = 'North', groupedBy=] RuntimeFilters: []



In [29]:
#Step 7 — Hidden Partitioning: The "Smart" Partition

With regular partitioning, if you partition by sale_date, a query WHERE sale_date = '2024-01-01' works. But WHERE year(sale_date) = 2024 does NOT prune — Iceberg doesn't know year() maps to your partition column. Hidden partitioning solves this.

In [31]:
from datetime import datetime

In [35]:
# Step 7: Hidden partitioning with transforms
spark.sql("DROP TABLE IF EXISTS local.db.events")
# spark.sql("""
#     CREATE TABLE local.db.events (
#         event_id    INT,
#         user_id     STRING,
#         event_type  STRING,
#         event_ts    TIMESTAMP
#     )
#     USING iceberg
#     PARTITIONED BY (
#         years(event_ts),      -- partition transform: extract year
#         months(event_ts)      -- partition transform: extract month
#     )
# """)


# events_data = [
#     (1, "u1", "click",    datetime(2024, 1, 15, 10, 0)),
#     (2, "u2", "purchase", datetime(2024, 1, 20, 14, 0)),
#     (3, "u3", "click",    datetime(2024, 6, 5,  9,  0)),
#     (4, "u4", "login",    datetime(2025, 1, 1,  0,  0)),
# ]
# df4 = spark.createDataFrame(events_data,
#       "event_id INT, user_id STRING, event_type STRING, event_ts TIMESTAMP")
# df4.writeTo("local.db.events").append()

# You’re hitting a known Iceberg restriction: you cannot define multiple time-based partition transforms (years, months, days, hours) on the same timestamp column in a single PARTITIONED BY clause, because Iceberg treats the less granular ones as redundant.

# Why this error happens
# Iceberg interprets years(event_ts) and months(event_ts) as overlapping information on the same column.

# The implementation explicitly rejects “redundant partition fields” like year + month + day on a single timestamp to avoid confusing partition layouts.

# That’s why your CREATE TABLE ... PARTITIONED BY (years(event_ts), months(event_ts)) fails with Cannot add redundant partition.




# warehouse/db/events/data/
# ├── event_ts_year=2024/
# │   ├── event_ts_month=2024-01/
# │   │   └── 00000-1-aaa.parquet   ← Jan 2024 events
# │   └── event_ts_month=2024-06/
# │       └── 00000-1-bbb.parquet   ← Jun 2024 events
# └── event_ts_year=2025/
#     └── event_ts_month=2025-01/
#         └── 00000-1-ccc.parquet   ← Jan 2025 events


# The magic: your QUERY doesn't mention partition columns at all!
# Iceberg automatically applies the year/month transform and prunes

# spark.sql("""
#     SELECT * FROM local.db.events 
#     WHERE event_ts >= '2024-01-01' AND event_ts < '2024-02-01'
# """).show()
# # ↑ Iceberg knows: year=2024, month=2024-01 → reads ONLY that one Parquet file

# spark.sql("SELECT partition, record_count FROM local.db.events.partitions").show()





DataFrame[]

In [37]:
# Step 8: Annotated write — understand every step

"""
WHAT HAPPENS WHEN YOU CALL df.writeTo("local.db.orders").append():

Step 1: Spark reads version-hint.txt → gets "3" → reads v3.metadata.json
        (learns the schema, partition spec, and current snapshot)

Step 2: Spark writes NEW Parquet data files to the data/ folder
        (these files exist but are "invisible" until committed)

Step 3: Spark writes a NEW manifest .avro file
        (records the new Parquet files + their column min/max/null stats)

Step 4: Spark writes a NEW manifest list .avro file  (= the snapshot)
        (records which manifests belong to this snapshot)

Step 5: Spark writes v4.metadata.json
        (adds the new snapshot to history, sets current-snapshot-id)

Step 6: Spark atomically updates version-hint.txt → "4"
        (THIS is the commit — if this fails, everything before is rolled back)
        (THIS is why Hadoop catalog is not safe for concurrent writers)

If Step 6 succeeds → all readers immediately see the new data
If Step 6 fails  → table is still at v3, new Parquet files become orphans
                    (cleaned up later by remove_orphan_files)
"""

'\nWHAT HAPPENS WHEN YOU CALL df.writeTo("local.db.orders").append():\n\nStep 1: Spark reads version-hint.txt → gets "3" → reads v3.metadata.json\n        (learns the schema, partition spec, and current snapshot)\n\nStep 2: Spark writes NEW Parquet data files to the data/ folder\n        (these files exist but are "invisible" until committed)\n\nStep 3: Spark writes a NEW manifest .avro file\n        (records the new Parquet files + their column min/max/null stats)\n\nStep 4: Spark writes a NEW manifest list .avro file  (= the snapshot)\n        (records which manifests belong to this snapshot)\n\nStep 5: Spark writes v4.metadata.json\n        (adds the new snapshot to history, sets current-snapshot-id)\n\nStep 6: Spark atomically updates version-hint.txt → "4"\n        (THIS is the commit — if this fails, everything before is rolled back)\n        (THIS is why Hadoop catalog is not safe for concurrent writers)\n\nIf Step 6 succeeds → all readers immediately see the new data\nIf Step 6

In [39]:
'''
    A catalog is just the “directory service” that tells engines, “for table db.orders, the current metadata file is here”. Different catalog types change where that pointer and table list live, not how snapshots/manifests/data work.

Main Iceberg catalog types
In Spark, you pick the catalog with configs like spark.sql.catalog.<name>.type. The common types are:

Type	Class behind it	Where metadata registry lives	Typical use
hadoop	SparkCatalog + HadoopCatalog	Filesystem directory (like your /warehouse)	Local/dev, simple S3/HDFS setups
hive	SparkCatalog + HiveCatalog	Hive Metastore database	Classic on-prem / EMR / CDP
rest	SparkCatalog + RESTCatalog	External REST service (e.g. Polaris, LakeKeeper)	Multi-engine, central catalog
glue	SparkCatalog + GlueCatalog	AWS Glue Data Catalog	AWS-native setups
jdbc	SparkCatalog + JdbcCatalog	Relational DB (Postgres, etc.)	Self-managed central catalog
nessie	SparkCatalog + NessieCatalog	Nessie server (Git-like branching)	Versioned catalogs (now being phased out)
Under the hood, all of them still use:
version-hint → vN.metadata.json → snap-*.avro → manifest-*.avro → Parquet; the catalog only controls where the pointer to vN.metadata.json lives and how concurrent writers coordinate.

What is SparkCatalog vs SparkSessionCatalog vs spark_catalog?
In Spark + Iceberg, the important classes/identifiers are:

org.apache.iceberg.spark.SparkCatalog

This is what you are using now:

python
.config("spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog")
.config("spark.sql.catalog.local.type", "hadoop")
.config("spark.sql.catalog.local.warehouse", WAREHOUSE_PATH)
It’s a generic front that delegates to a specific catalog implementation based on type=hadoop | hive | rest | glue | jdbc | nessie. 

You then reference tables as local.db.orders:

local → the catalog name

db → namespace (schema)

orders → table name

org.apache.iceberg.spark.SparkSessionCatalog

This wraps Spark’s built-in catalog (what Spark uses by default for spark_catalog / default / hive_metastore).

It lets you say: “treat all spark_catalog tables as Iceberg by default”, while still allowing non-Iceberg tables:

bash
spark.sql.catalog.spark_catalog = org.apache.iceberg.spark.SparkSessionCatalog
spark.sql.catalog.spark_catalog.type = hive
Then spark.sql("SELECT * FROM db.orders") automatically goes through Iceberg if the table is an Iceberg table in that metastore.

spark_catalog (plain Spark, no Iceberg)

This is Spark’s default session catalog name.

Without Iceberg, spark_catalog typically maps to the built-in Hive or in-memory catalog.

With SparkSessionCatalog, Iceberg plugs into that, so:

Iceberg tables stored in Hive metastore are handled by Iceberg

Non-Iceberg tables are delegated back to Spark’s default behavior.

So:

You: use a separate Iceberg catalog called local via SparkCatalog + type=hadoop (local warehouse folder).

Production: often uses SparkCatalog + type=hive or type=glue and sometimes replaces spark_catalog with SparkSessionCatalog so SELECT * FROM db.tbl “just works” across the org.

How they differ from the Hadoop catalog you’re using
Right now, your config:

python
spark.sql.catalog.local = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.local.type = hadoop
spark.sql.catalog.local.warehouse = /.../warehouse
Meaning:

The mapping “local.db.orders → current metadata file path” is stored as:

A directory layout under warehouse/db/orders/metadata

With version-hint.txt as the pointer (not strongly atomic).

Great for you learning on a single laptop, not safe for multiple writers in prod.

If you switch to a Hive or Glue catalog, configs look like:

bash
# Hive-backed Iceberg catalog
spark.sql.catalog.prod = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.prod.type = hive
spark.sql.catalog.prod.uri  = thrift://hive-metastore:9083
spark.sql.catalog.prod.warehouse = s3://my-bucket/warehouse
bash
# Glue-backed (AWS) Iceberg catalog
spark.sql.catalog.prod = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.prod.type = glue
spark.sql.catalog.prod.warehouse = s3://my-bucket/warehouse
Then prod.db.orders will store the “current metadata file” pointer in Hive/Glue, which can do atomic, transactional updates, so many Spark jobs can safely write concurrently.

Interview-level summary you can give
“Iceberg catalog” = service that maps table name → current metadata file and manages table list, namespaces, and ACID-style commit semantics.

Locally I used Hadoop catalog via SparkCatalog (pointer lives as files + version-hint.txt), ideal for single-writer learning.

In prod you’d typically use Hive, Glue, REST or JDBC catalogs, still via SparkCatalog, so commit pointers live in a metastore or service that supports atomic updates and multi-writer coordination.

SparkSessionCatalog lets Iceberg plug into Spark’s built‑in spark_catalog so existing queries (SELECT * FROM db.tbl) transparently use Iceberg where tables are Iceberg.
    
    
'''

'\n    A catalog is just the “directory service” that tells engines, “for table db.orders, the current metadata file is here”. Different catalog types change where that pointer and table list live, not how snapshots/manifests/data work.\n\nMain Iceberg catalog types\nIn Spark, you pick the catalog with configs like spark.sql.catalog.<name>.type. The common types are:\n\nType\tClass behind it\tWhere metadata registry lives\tTypical use\nhadoop\tSparkCatalog + HadoopCatalog\tFilesystem directory (like your /warehouse)\tLocal/dev, simple S3/HDFS setups\nhive\tSparkCatalog + HiveCatalog\tHive Metastore database\tClassic on-prem / EMR / CDP\nrest\tSparkCatalog + RESTCatalog\tExternal REST service (e.g. Polaris, LakeKeeper)\tMulti-engine, central catalog\nglue\tSparkCatalog + GlueCatalog\tAWS Glue Data Catalog\tAWS-native setups\njdbc\tSparkCatalog + JdbcCatalog\tRelational DB (Postgres, etc.)\tSelf-managed central catalog\nnessie\tSparkCatalog + NessieCatalog\tNessie server (Git-like branch

In [40]:
# Step 9: Understand how Iceberg plans a query (2-level pruning)

"""
QUERY: SELECT * FROM sales WHERE region = 'North' AND amount > 400

HOW ICEBERG PLANS THIS — WITHOUT LISTING DIRECTORIES:

Level 1 — Manifest List pruning (partition level):
  → Read the current manifest list (.avro)
  → Each manifest entry has partition_lower_bound / partition_upper_bound
  → Skip manifests where region_partition range doesn't include 'North'
  → Result: only 1 manifest file to read (instead of 3)

Level 2 — Manifest File pruning (file level):
  → Read the surviving manifest file
  → Each data file entry has lower_bounds{amount: 100}, upper_bounds{amount: 700}
  → Check if upper_bound(amount) > 400 → keep files that MIGHT have rows
  → Check if lower_bound(amount) > 400 → skip files where ALL rows fail the filter
  → Result: only 1 Parquet file read

Level 3 — Parquet internal row-group pruning (row group level):
  → Inside the Parquet file itself, each 'row group' also has min/max stats
  → Spark skips row groups that can't satisfy amount > 400

This 3-level pruning is why Iceberg queries on TB-scale tables take seconds.
"""

# See this in action with explain()
spark.sql("""
    SELECT * FROM local.db.sales WHERE region = 'North' AND amount > 400
""").explain("formatted")
# Look for: IcebergScanRelation, PartitionFilters, PushedFilters


== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- BatchScan local.db.sales (1)


(1) BatchScan local.db.sales
Output [4]: [sale_id#1241, region#1242, amount#1243, sale_date#1244]
local.db.sales (branch=null) [filters=region IS NOT NULL, amount IS NOT NULL, region = 'North', amount > 400.0, groupedBy=]

(2) ColumnarToRow [codegen id : 1]
Input [4]: [sale_id#1241, region#1242, amount#1243, sale_date#1244]

(3) Filter [codegen id : 1]
Input [4]: [sale_id#1241, region#1242, amount#1243, sale_date#1244]
Condition : (isnotnull(amount#1243) AND (amount#1243 > 400.0))




In [41]:
# warehouse/
# ├── db/
# │   ├── orders/
# │   │   ├── metadata/
# │   │   │   ├── version-hint.txt          ← "4" (Hadoop catalog pointer)
# │   │   │   ├── v1.metadata.json          ← CREATE TABLE  (schema only)
# │   │   │   ├── v2.metadata.json          ← After 1st INSERT
# │   │   │   ├── v3.metadata.json          ← After 2nd INSERT
# │   │   │   ├── v4.metadata.json          ← After 3rd INSERT (CURRENT)
# │   │   │   ├── snap-AAA.avro             ← Manifest List = Snapshot 1
# │   │   │   ├── snap-BBB.avro             ← Manifest List = Snapshot 2
# │   │   │   └── snap-CCC.avro             ← Manifest List = Snapshot 3
# │   │   └── data/
# │   │       ├── 00000-1-xxx.parquet       ← Parquet: 3 rows (Snappy compressed)
# │   │       ├── 00000-2-yyy.parquet       ← Parquet: 2 rows
# │   │       └── 00000-3-zzz.parquet       ← Parquet: new rows
# │   ├── sales/
# │   │   └── data/
# │   │       ├── region=North/...          ← Partitioned by region
# │   │       ├── region=South/...
# │   │       └── region=East/...
# │   └── events/
# │       └── data/
# │           └── event_ts_year=2024/
# │               └── event_ts_month=2024-01/...   ← Hidden partition transforms


In [42]:
# CELL 4 — Schema Evolution: Add, Rename, Drop columns

'''

# --- 4a. ADD a new column ---
spark.sql("ALTER TABLE local.db.employees ADD COLUMN bonus DOUBLE")
spark.sql("SELECT * FROM local.db.employees").show()
# Notice: existing rows show NULL for bonus — old data files are untouched ✅

# --- 4b. ADD another column with a default position ---
spark.sql("ALTER TABLE local.db.employees ADD COLUMN manager_id INT AFTER emp_id")

# --- 4c. RENAME a column (this is IMPOSSIBLE in Hive — Iceberg makes it trivial) ---
spark.sql("ALTER TABLE local.db.employees RENAME COLUMN manager_id TO reports_to")

# --- 4d. TYPE PROMOTION — widen a type safely (INT → BIGINT is safe, BIGINT → INT is NOT) ---
spark.sql("ALTER TABLE local.db.employees ALTER COLUMN emp_id TYPE bigint")

# --- 4e. DROP a column ---
spark.sql("ALTER TABLE local.db.employees DROP COLUMN reports_to")

# See the evolved schema
spark.sql("DESCRIBE TABLE local.db.employees").show()

# 🧠 Key Interview Point:
# Iceberg uses column IDs internally (not names), so renames don't break old readers.
# When an old file is read, Iceberg maps column IDs to the new schema automatically.
print("\n✅ Schema evolved — zero data rewrite happened!")

'''


'\n\n# --- 4a. ADD a new column ---\nspark.sql("ALTER TABLE local.db.employees ADD COLUMN bonus DOUBLE")\nspark.sql("SELECT * FROM local.db.employees").show()\n# Notice: existing rows show NULL for bonus — old data files are untouched ✅\n\n# --- 4b. ADD another column with a default position ---\nspark.sql("ALTER TABLE local.db.employees ADD COLUMN manager_id INT AFTER emp_id")\n\n# --- 4c. RENAME a column (this is IMPOSSIBLE in Hive — Iceberg makes it trivial) ---\nspark.sql("ALTER TABLE local.db.employees RENAME COLUMN manager_id TO reports_to")\n\n# --- 4d. TYPE PROMOTION — widen a type safely (INT → BIGINT is safe, BIGINT → INT is NOT) ---\nspark.sql("ALTER TABLE local.db.employees ALTER COLUMN emp_id TYPE bigint")\n\n# --- 4e. DROP a column ---\nspark.sql("ALTER TABLE local.db.employees DROP COLUMN reports_to")\n\n# See the evolved schema\nspark.sql("DESCRIBE TABLE local.db.employees").show()\n\n# 🧠 Key Interview Point:\n# Iceberg uses column IDs internally (not names), so renames

In [43]:
# CELL 7 — Maintenance: Compaction, Snapshot Expiry, Orphan Cleanup


'''
# --- Write many small files to simulate the small file problem ---
for i in range(5):
    tiny_data = [(i*10+j, f"prod_{j}", float(j*10)) for j in range(3)]
    spark.createDataFrame(tiny_data).toDF("id","product","price") \
         .writeTo("local.db.orders").append()

print("Files BEFORE compaction:")
spark.sql("SELECT file_path, record_count FROM local.db.orders.files").show(truncate=False)

# --- 7a. COMPACTION (rewrite_data_files) — merges small files into larger ones ---
spark.sql("""
    CALL local.system.rewrite_data_files(
        table => 'db.orders',
        strategy => 'binpack',          -- packs files together
        options => map('target-file-size-bytes', '134217728')  -- 128MB target
    )
""").show()
print("\n✅ After compaction — fewer, larger files:")
spark.sql("SELECT file_path, record_count FROM local.db.orders.files").show(truncate=False)

# --- 7b. SNAPSHOT EXPIRY — delete old snapshot metadata (frees disk space) ---
spark.sql("""
    CALL local.system.expire_snapshots(
        table => 'db.orders',
        older_than => TIMESTAMP '2099-01-01 00:00:00',  -- expire all for demo
        retain_last => 1                                 -- but always keep latest
    )
""").show()
print("✅ Old snapshots expired")

# --- 7c. ORPHAN FILE CLEANUP — delete data files not referenced by any snapshot ---
spark.sql("""
    CALL local.system.remove_orphan_files(table => 'db.orders')
""").show()
print("✅ Orphan files removed")

'''


'\n# --- Write many small files to simulate the small file problem ---\nfor i in range(5):\n    tiny_data = [(i*10+j, f"prod_{j}", float(j*10)) for j in range(3)]\n    spark.createDataFrame(tiny_data).toDF("id","product","price")          .writeTo("local.db.orders").append()\n\nprint("Files BEFORE compaction:")\nspark.sql("SELECT file_path, record_count FROM local.db.orders.files").show(truncate=False)\n\n# --- 7a. COMPACTION (rewrite_data_files) — merges small files into larger ones ---\nspark.sql("""\n    CALL local.system.rewrite_data_files(\n        table => \'db.orders\',\n        strategy => \'binpack\',          -- packs files together\n        options => map(\'target-file-size-bytes\', \'134217728\')  -- 128MB target\n    )\n""").show()\nprint("\n✅ After compaction — fewer, larger files:")\nspark.sql("SELECT file_path, record_count FROM local.db.orders.files").show(truncate=False)\n\n# --- 7b. SNAPSHOT EXPIRY — delete old snapshot metadata (frees disk space) ---\nspark.sql("""\